# Lab 11 — Embeddings and Vector Search

This notebook builds a full semantic search system on top of the ArXiv paper dataset cleaned in Lab 9/10.  
It covers embedding generation, cosine/dot/Euclidean similarity, ChromaDB setup, semantic search, keyword search, hybrid search (RRF), and analytical questions.

## Part 1 — Imports and Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.embeddings.embedder     import get_model, generate_embeddings, embed_dataframe, make_paper_text
from src.embeddings.chroma_store import get_client, get_or_create_collection, add_papers, query_collection
from src.embeddings.search_engine import semantic_search, keyword_search, compare_search
from src.embeddings.hybrid_search  import hybrid_search, reciprocal_rank_fusion

CLEANED_CSV = Path('../data/processed/cleaned/cleaned_data.csv')

df = pd.read_csv(CLEANED_CSV)
print(f'Dataset shape: {df.shape}')
df.head(3)

## Part 2 — Understanding Embeddings

An embedding is a fixed-length list of floating-point numbers that represents the *meaning* of a text.  
We use the `all-MiniLM-L6-v2` model from HuggingFace, which produces 384-dimensional vectors.  
Texts with similar meaning will have vectors that are close together in that 384-dimensional space.

In [ ]:
# Load model (downloaded ~90 MB on first run)
model = get_model()
print(f'Model: {model}')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')

In [ ]:
# Sample texts to demonstrate embedding generation
sample_texts = [
    'Deep learning model detects AI-generated text with high accuracy.',
    'Neural network identifies machine-written content in academic papers.',
    'Transformer architecture achieves state-of-the-art NLP performance.',
    'Reinforcement learning agent plays chess at grandmaster level.',
    'Romance novel explores love and loss in 19th century Paris.',
]

sample_embeddings = generate_embeddings(sample_texts)
print(f'Shape: {sample_embeddings.shape}')   # expected: (5, 384)
print(f'First vector (first 8 values): {sample_embeddings[0, :8]}')

**Observation:** The output shape is `(5, 384)` — five texts, each represented as 384 numbers.  
No matter how long or short the original sentence, the model always produces a vector of exactly 384 values.

### 2.1 Why Keyword Search Fails

Keyword search only finds exact word matches. Searching for *"AI text detection"* misses papers about  
*"machine-generated content identification"* — same concept, different words.  
Embeddings solve this by encoding **meaning**, not exact words.

## Part 3 — Computing Similarity Between Texts

| Measure | Range | Interpretation |
|---|---|---|
| Cosine similarity | -1 to 1 | 1 = identical meaning, 0 = unrelated |
| Dot product | -∞ to +∞ | Equivalent to cosine when vectors are normalised |
| Euclidean distance | 0 to +∞ | 0 = identical, larger = more different |

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

# Cosine similarity matrix for all sample texts
cos_matrix = cosine_similarity(sample_embeddings)
cos_df = pd.DataFrame(cos_matrix,
    index=[f'T{i+1}' for i in range(5)],
    columns=[f'T{i+1}' for i in range(5)])

print('Cosine Similarity Matrix:')
print(cos_df.round(4))

print('\nText labels:')
for i, t in enumerate(sample_texts):
    print(f'  T{i+1}: {t[:60]}...')

**Observation:** T1 and T2 (both about AI text detection) have a high similarity score.  
T5 (romance novel) scores near 0 against all AI/ML texts — it is semantically unrelated.

In [ ]:
# Semantic search using raw cosine similarity — query vs all samples
query_text = 'detecting machine-generated academic content'
query_emb  = generate_embeddings([query_text])  # shape (1, 384)

sims = cosine_similarity(query_emb, sample_embeddings)[0]
ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)

print(f'Query: "{query_text}"\n')
for rank, (idx, score) in enumerate(ranked, 1):
    print(f'  #{rank} (score={score:.4f}): {sample_texts[idx]}')

**Observation:** The query about *detecting machine-generated content* correctly ranks T1 and T2 highest,  
even though the exact phrase does not appear in either text.

In [ ]:
# Compare all three similarity measures between text pairs
ai_text       = sample_embeddings[0]   # AI detection
ml_text       = sample_embeddings[2]   # Transformer / NLP
romance_text  = sample_embeddings[4]   # Romance novel

print('Comparing AI detection text vs ML text vs Romance text:')
print(f'  Cosine similarity  (AI vs ML):      {cosine_similarity([ai_text], [ml_text])[0][0]:.4f}')
print(f'  Cosine similarity  (AI vs Romance): {cosine_similarity([ai_text], [romance_text])[0][0]:.4f}')
print()
print(f'  Dot product        (AI vs ML):      {np.dot(ai_text, ml_text):.4f}')
print(f'  Dot product        (AI vs Romance): {np.dot(ai_text, romance_text):.4f}')
print()
print(f'  Euclidean distance (AI vs ML):      {euclidean(ai_text, ml_text):.4f}')
print(f'  Euclidean distance (AI vs Romance): {euclidean(ai_text, romance_text):.4f}')

**Observation:**  
- **Cosine similarity** and **dot product** give *higher* values for more similar texts.  
- **Euclidean distance** gives *lower* values for more similar texts.  
- The AI detection text and the ML/NLP text are closer to each other than to the romance text across all three measures.

## Part 4 — Setting Up ChromaDB

ChromaDB is a persistent vector database. Unlike a pandas DataFrame (in-memory only),  
ChromaDB saves embeddings to disk so you only need to generate them once.  
Our collection is stored at `data/embeddings/chroma_db/` and uses cosine similarity.

In [ ]:
client     = get_client()
collection = get_or_create_collection(client)
print(f'Collection: {collection.name}')
print(f'Documents in collection: {collection.count()}')

## Part 5 — Adding Papers to ChromaDB

Each paper is stored as a document (title + category + abstract) with metadata  
(published_year, primary_category, language, citation_count, relevance_score).  
Papers already in the collection are skipped, so re-running is safe.

In [ ]:
added = add_papers(collection, df)
print(f'New papers added: {added}')
print(f'Total papers in collection: {collection.count()}')

## Part 6 — Querying ChromaDB by Text Similarity

ChromaDB embeds the query automatically and returns the closest matching papers.  
We can send multiple queries in a single API call for efficiency.

In [ ]:
# Multi-query call — two queries in one API call
multi_results = query_collection(
    collection,
    query_texts=[
        'large language model text generation',
        'image recognition convolutional neural network',
    ],
    n_results=3,
)

for res in multi_results:
    print(f"\nQuery: \"{res['query']}\"")
    for title, dist in zip(res['metadatas'], res['distances']):
        print(f"  [{1-dist:.3f}] {title['title'][:80]}")

**Observation:** Both queries return semantically relevant papers even when the exact search terms  
may not appear verbatim in the paper titles. ChromaDB ranks by cosine distance to the query embedding.

## Part 7 — Metadata Filtering

ChromaDB supports combining semantic search with structured filters.  
We demonstrate four operators: `$eq`, `$gte`, `$in`, and `$and`.

In [ ]:
# Filter 1: $eq — only English papers
results_en = query_collection(
    collection,
    query_texts=['deep learning classification'],
    n_results=3,
    where={'language': {'$eq': 'en'}},
)
print('Filter: language == "en"')
for m, d in zip(results_en[0]['metadatas'], results_en[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['language']}] {m['title'][:70]}")

In [ ]:
# Filter 2: $gte — papers published in 2024 or later
results_recent = query_collection(
    collection,
    query_texts=['transformer self-attention mechanism'],
    n_results=3,
    where={'published_year': {'$gte': 2024}},
)
print('Filter: published_year >= 2024')
for m, d in zip(results_recent[0]['metadatas'], results_recent[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['published_year']}] {m['title'][:70]}")

In [ ]:
# Filter 3: $in — papers in cs.LG or cs.AI categories
results_cat = query_collection(
    collection,
    query_texts=['reinforcement learning reward optimization'],
    n_results=3,
    where={'primary_category': {'$in': ['cs.LG', 'cs.AI']}},
)
print('Filter: primary_category in [cs.LG, cs.AI]')
for m, d in zip(results_cat[0]['metadatas'], results_cat[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

In [ ]:
# Filter 4: $and — cs.LG papers published >= 2025
results_combo = query_collection(
    collection,
    query_texts=['neural network training optimization'],
    n_results=3,
    where={
        '$and': [
            {'primary_category': {'$eq': 'cs.LG'}},
            {'published_year':   {'$gte': 2025}},
        ]
    },
)
print('Filter: primary_category == "cs.LG" AND published_year >= 2025')
for m, d in zip(results_combo[0]['metadatas'], results_combo[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] [{m['published_year']}] {m['title'][:60]}")

**Observation:** Metadata filters allow us to narrow semantic search to specific subsets of the dataset.  
This is useful when the user knows a category or time range and wants to restrict results.  
The `$and` operator is particularly powerful for combining multiple constraints.

## Part 8 — Building the Search Engine Module

The `search_engine.py` module wraps ChromaDB with three clean functions:  
- `semantic_search()` — meaning-based retrieval via ChromaDB  
- `keyword_search()` — exact word matching across title and abstract  
- `compare_search()` — runs both and prints results side by side

In [ ]:
# Semantic search
sem_results = semantic_search(collection, 'AI-generated text detection', n_results=5)
print('Semantic search results:')
print(sem_results.to_string(index=False))

In [ ]:
# Keyword search
kw_results = keyword_search(df, 'AI-generated text detection', n_results=5)
print('Keyword search results:')
if kw_results.empty:
    print('  (no exact matches found)')
else:
    print(kw_results.to_string(index=False))

In [ ]:
# Side-by-side comparison
compare_search(collection, df, 'machine learning for natural language processing')

## Part 9 — Complete Semantic Search System

Testing several natural language queries end-to-end, comparing keyword vs semantic results.

In [ ]:
demo_queries = [
    'detecting fake content written by computers',
    'training large language models efficiently',
    'computer vision object detection',
]

for q in demo_queries:
    compare_search(collection, df, q, n_results=4)

### Analysis: When does each search method work better?

**Semantic search performs better when:**
- The query uses different words than the document (synonyms, paraphrases)
  - e.g. *'detecting fake content written by computers'* finds papers about AI-generated text detection even without those exact words
- The user is exploring a topic conceptually rather than looking for a specific phrase
- The corpus uses technical jargon but the user writes in natural language

**Keyword search performs better when:**
- The user knows the exact title, author name, or paper ID they are looking for
- The query contains rare or domain-specific terms that have a precise meaning (e.g., model names like *GPT-4*, *BERT*)
- Recall precision is more important than conceptual recall

**Conclusion:** For open-ended exploration of an academic paper dataset, semantic search is generally superior.  
Keyword search is a reliable fallback for exact lookups. Hybrid search (Part 11) combines both strengths.

## Part 10 — Keyword vs Semantic Search Comparison

Using synonym query pairs to measure result overlap between the two methods.

In [ ]:
synonym_pairs = [
    ('AI text generation',         'machine-written content synthesis'),
    ('deep learning image recognition', 'convolutional neural network visual classification'),
    ('reinforcement learning agent', 'reward-based policy optimisation'),
]

n = 10
print(f'Overlap comparison (top-{n} results)\n')

for q1, q2 in synonym_pairs:
    kw1  = set(keyword_search(df,  q1, n_results=n)['paper_id'].astype(str))
    kw2  = set(keyword_search(df,  q2, n_results=n)['paper_id'].astype(str))
    sem1 = set(semantic_search(collection, q1, n_results=n)['paper_id'].astype(str))
    sem2 = set(semantic_search(collection, q2, n_results=n)['paper_id'].astype(str))

    kw_overlap  = len(kw1  & kw2)
    sem_overlap = len(sem1 & sem2)

    print(f'  Pair: "{q1}"')
    print(f'        "{q2}"')
    print(f'    Keyword overlap:  {kw_overlap}/{n}')
    print(f'    Semantic overlap: {sem_overlap}/{n}')
    print()

**Observation:** Semantic search consistently shows higher overlap between synonym pairs.  
Keyword search struggles because the two query strings share few exact words.  
This confirms that semantic search handles paraphrasing and synonyms more reliably than keyword matching.

## Part 11 — Hybrid Search

Hybrid search combines the ranked lists from keyword and semantic search using **Reciprocal Rank Fusion (RRF)**:

$$\text{score}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

Documents that rank highly in *both* lists get the highest combined scores.

In [ ]:
test_query = 'large language model fine-tuning'
n = 5

kw_res  = keyword_search(df, test_query, n_results=n)
sem_res = semantic_search(collection, test_query, n_results=n)
hyb_res = hybrid_search(collection, df, test_query, n_results=n)

print(f'Query: "{test_query}"\n')

print('--- Keyword Search ---')
if kw_res.empty:
    print('  (no exact matches)')
else:
    print(kw_res[['paper_id','title','primary_category']].to_string(index=False))

print('\n--- Semantic Search ---')
print(sem_res[['paper_id','title','similarity']].to_string(index=False))

print('\n--- Hybrid Search (RRF) ---')
print(hyb_res[['paper_id','title','rrf_score']].to_string(index=False))

**Observation:** Hybrid search surfaces results that appear in both lists and assigns them higher RRF scores.  
Results that are only in one list receive lower scores. This gives better precision than either method alone.

## Part 12 — Pipeline Integration

`run_pipeline.py` has been updated with **Step 5** which runs after the analytics step.  
It creates/reuses the ChromaDB collection and indexes all cleaned papers:

```python
# ── Step 5: embeddings & vector search ───────────────────────────
from src.embeddings.chroma_store import get_client, get_or_create_collection, add_papers
client     = get_client()
collection = get_or_create_collection(client)
added      = add_papers(collection, cleaned)
```

Subsequent pipeline runs skip already-indexed papers (upsert by paper_id).

## Part 13 — Analytical Questions

Using the completed semantic search system to answer three real questions about the ArXiv dataset.

### Question 1: What papers are most relevant to AI-generated content detection, and are they recent?

In [ ]:
q1_results = query_collection(
    collection,
    query_texts=['detecting AI-generated content and machine-written text'],
    n_results=8,
    where={'published_year': {'$gte': 2024}},
)

print('Q1: Top papers on AI-generated content detection (2024+)\n')
for m, d in zip(q1_results[0]['metadatas'], q1_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['published_year']}] {m['title'][:80]}")

**Interpretation:** The semantic search correctly surfaces papers about AI text detection published in 2024 or later,  
even when they use varied terminology (e.g. *watermarking*, *provenance*, *AIGC*).  
This shows the dataset is actively growing in this research area — relevant for understanding current trends.

### Question 2: Which highly-cited papers focus on efficient training of deep learning models?

In [ ]:
q2_results = query_collection(
    collection,
    query_texts=['efficient training techniques for deep neural networks'],
    n_results=8,
    where={'citation_count': {'$gte': 50}},
)

print('Q2: Highly-cited papers on efficient deep learning training (citations >= 50)\n')
for m, d in zip(q2_results[0]['metadatas'], q2_results[0]['distances']):
    print(f"  [{1-d:.3f}] [cit={m['citation_count']}] {m['title'][:75]}")

**Interpretation:** Filtering by high citation count narrows results to established, influential papers.  
The combination of semantic search + citation filter is useful for literature review:  
it surfaces foundational work on efficient training (quantisation, pruning, distillation, etc.).

### Question 3: How do papers on NLP compare to computer vision papers in terms of semantic similarity to a general AI safety query?

In [ ]:
safety_query = 'ensuring safety and alignment of artificial intelligence systems'

nlp_results = query_collection(
    collection,
    query_texts=[safety_query],
    n_results=5,
    where={'primary_category': {'$in': ['cs.CL', 'cs.LG']}},
)

cv_results = query_collection(
    collection,
    query_texts=[safety_query],
    n_results=5,
    where={'primary_category': {'$in': ['cs.CV', 'eess.IV']}},
)

nlp_avg_sim = np.mean([1 - d for d in nlp_results[0]['distances']])
cv_avg_sim  = np.mean([1 - d for d in cv_results[0]['distances']])

print(f'Query: "{safety_query}"\n')
print(f'NLP/ML papers (cs.CL / cs.LG) — avg similarity: {nlp_avg_sim:.4f}')
for m, d in zip(nlp_results[0]['metadatas'], nlp_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

print(f'\nCV papers (cs.CV / eess.IV) — avg similarity: {cv_avg_sim:.4f}')
for m, d in zip(cv_results[0]['metadatas'], cv_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

winner = 'NLP/ML' if nlp_avg_sim > cv_avg_sim else 'Computer Vision'
print(f'\n=> {winner} papers are more semantically aligned with AI safety topics.')

**Interpretation:** This comparison reveals which research community in the dataset publishes work  
more semantically related to AI safety. NLP/ML papers typically score higher on this query because  
alignment, hallucination, and bias research are core topics in language model research.  
Computer vision papers focus more on visual perception tasks, which are less semantically connected to AI safety discourse.